# RECELL-AI — XGBoost SOH Training (NASA Battery Dataset)

Notebook produksi untuk melatih `soh_xgb_model.json` yang dipakai di `jetson/src/main.py`.

**Dataset:** [`patrickfleith/nasa-battery-dataset`](https://www.kaggle.com/datasets/patrickfleith/nasa-battery-dataset) — cleaned CSV format. Data di-load via `metadata.csv` + per-cycle CSV files.

**Target baterai:** B0005, B0006, B0007, B0018 (NASA Ames PCoE, 18650 Li-Ion)

**Kontrak fitur (WAJIB sama dengan `main.py:140`):** `v_drop`, `internal_r`, `temp_delta`

**Evaluasi:** Leave-One-Battery-Out 4-fold

**Output:** `soh_xgb_model.json` (auto-download)

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn kagglehub

## Cell 2 — Authenticate Kaggle

Pakai token API format baru (`KAGGLE_API_TOKEN=KGAT_...`).
Generate token di https://www.kaggle.com/settings → scroll ke *API* → *Create New Token*.

Sel di bawah pakai `getpass` — token Anda **tidak** akan masuk ke output cell, log, atau notebook tersimpan.

In [ ]:
import os
import getpass

TOKEN_FILE = os.path.expanduser('~/.kaggle/access_token')

if not os.environ.get('KAGGLE_API_TOKEN') and not os.path.exists(TOKEN_FILE):
    token = getpass.getpass('Paste Kaggle API token (KGAT_...): ').strip()
    assert token.startswith('KGAT_'), 'Token should start with KGAT_'
    os.environ['KAGGLE_API_TOKEN'] = token
    os.makedirs(os.path.dirname(TOKEN_FILE), exist_ok=True)
    with open(TOKEN_FILE, 'w') as f:
        f.write(token)
    os.chmod(TOKEN_FILE, 0o600)
    print('[OK] Token saved to env + ~/.kaggle/access_token')
else:
    print('[OK] Kaggle token already configured.')

## Cell 3 — Download Dataset & Load Metadata

Dataset `patrickfleith/nasa-battery-dataset` berisi cleaned CSV files.
Struktur: `cleaned_dataset/metadata.csv` + `cleaned_dataset/data/XXXXX.csv` per cycle.
Filter metadata untuk 4 baterai target.

In [ ]:
import kagglehub
import pandas as pd
from pathlib import Path

TARGET_BATTERIES = ['B0005', 'B0006', 'B0007', 'B0018']

dataset_root = Path(kagglehub.dataset_download('patrickfleith/nasa-battery-dataset'))
data_dir = dataset_root / 'cleaned_dataset'
print(f'[*] Dataset root: {dataset_root}')
print(f'[*] Data dir:     {data_dir}\n')

meta = pd.read_csv(data_dir / 'metadata.csv')
meta_target = meta[meta['battery_id'].isin(TARGET_BATTERIES)].copy()

assert len(meta_target) > 0, (
    f'No records found for {TARGET_BATTERIES} in metadata.csv.\n'
    f'Available battery_ids: {sorted(meta["battery_id"].unique())}'
)

print(f'[*] Found {len(meta_target)} records for {sorted(meta_target["battery_id"].unique())}')
print(f'    Types: {meta_target["type"].value_counts().to_dict()}')

## Cell 4 — Load Discharge Cycles → Per-Cycle DataFrame

Filter `type == 'discharge'` cycles yang punya nilai `Capacity`.
Load tiap CSV file dan ambil array `Voltage_measured`, `Current_measured`, `Temperature_measured`.

In [ ]:
import numpy as np

rows = []
discharge_meta = meta_target[
    (meta_target['type'] == 'discharge') & (meta_target['Capacity'].notna())
].copy()

for _, row in discharge_meta.iterrows():
    csv_path = data_dir / 'data' / row['filename']
    if not csv_path.exists():
        continue
    df = pd.read_csv(csv_path)

    v = df['Voltage_measured'].values
    i = df['Current_measured'].values
    t = df['Temperature_measured'].values

    rows.append({
        'battery_id':   row['battery_id'],
        'cycle_id':     row['test_id'],
        'voltages':     v,
        'currents':     i,
        'temperatures': t,
        'capacity_ah':  row['Capacity'],
    })

cycles_raw = pd.DataFrame(rows)
print(f'[*] Total discharge cycles loaded: {len(cycles_raw)}')
print(cycles_raw['battery_id'].value_counts().sort_index())

## Cell 5 — Feature Engineering

Ekstrak 3 fitur per cycle (formula PERSIS sama dengan `main.py:135-140`):

| Fitur          | Rumus                                |
|----------------|--------------------------------------|
| `v_drop`       | `max(V) - min(V)`                    |
| `internal_r`   | `v_drop / abs(min(I))`               |
| `temp_delta`   | `max(T) - T[0]`                      |
| `soh` (target) | `clip((Capacity / 2.0) * 100, 0, 100)` |

Skip cycle dengan `len(voltages) < 10` atau `i_max == 0`.

In [ ]:
NOMINAL_CAPACITY_AH = 2.0  # matches production SoH semantics

feat_rows = []
for _, c in cycles_raw.iterrows():
    v, i, t = c['voltages'], c['currents'], c['temperatures']

    if len(v) < 10:
        continue
    i_max = abs(np.min(i))
    if i_max == 0:
        continue

    v_drop     = float(np.max(v) - np.min(v))
    internal_r = float(v_drop / i_max)
    temp_delta = float(np.max(t) - t[0])
    soh        = float(np.clip((c['capacity_ah'] / NOMINAL_CAPACITY_AH) * 100, 0, 100))

    feat_rows.append({
        'battery_id': c['battery_id'],
        'cycle_id':   c['cycle_id'],
        'v_drop':     v_drop,
        'internal_r': internal_r,
        'temp_delta': temp_delta,
        'soh':        soh,
    })

feat = pd.DataFrame(feat_rows)
print(f'[*] Extracted features for {len(feat)} cycles')
print(feat.describe().round(3))
feat.head()

## Cell 6 — Exploratory Data Analysis

Sanity check: SoH harus turun seiring umur, fitur harus berkorelasi dengan SoH.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# (a) SoH degradation per battery
for bid, g in feat.groupby('battery_id'):
    g_sorted = g.sort_values('cycle_id')
    axes[0].plot(range(len(g_sorted)), g_sorted['soh'].values, label=bid)
axes[0].set_title('SoH vs Cycle index (per battery)')
axes[0].set_xlabel('cycle index (sorted)'); axes[0].set_ylabel('SoH (%)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# (b) Feature distributions
for col, color in zip(['v_drop', 'internal_r', 'temp_delta'], ['C0', 'C1', 'C2']):
    axes[1].hist(feat[col], bins=30, alpha=0.5, label=col, color=color)
axes[1].set_title('Feature distributions')
axes[1].set_xlabel('value'); axes[1].set_ylabel('count')
axes[1].legend(); axes[1].grid(alpha=0.3)

# (c) Correlation heatmap
corr = feat[['v_drop', 'internal_r', 'temp_delta', 'soh']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[2])
axes[2].set_title('Feature ↔ SoH correlation')

plt.tight_layout(); plt.show()

## Cell 7 — Train & Evaluate (Leave-One-Battery-Out CV)

`GroupKFold(n_splits=4, groups=battery_id)` — tiap fold: train di 3 baterai, test di 1.
Laporkan RMSE & MAE per fold + rata-rata. Plot prediksi vs aktual untuk tiap baterai test.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

FEATURES = ['v_drop', 'internal_r', 'temp_delta']  # MUST match main.py:140 order
TARGET   = 'soh'

X = feat[FEATURES]
y = feat[TARGET]
groups = feat['battery_id']

HYPERPARAMS = dict(
    objective='reg:squarederror',
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    random_state=42,
)

gkf = GroupKFold(n_splits=4)
fold_metrics = []
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    test_battery = groups.iloc[test_idx].unique()[0]
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    fold_model = xgb.XGBRegressor(**HYPERPARAMS)
    fold_model.fit(X_tr, y_tr)
    y_pred = fold_model.predict(X_te)

    rmse = float(np.sqrt(mean_squared_error(y_te, y_pred)))
    mae  = float(mean_absolute_error(y_te, y_pred))
    fold_metrics.append({'fold': fold, 'test_battery': test_battery, 'rmse': rmse, 'mae': mae})
    print(f'  Fold {fold} — test={test_battery}  RMSE={rmse:.2f}%  MAE={mae:.2f}%')

    ax = axes[fold // 2, fold % 2]
    ax.scatter(y_te, y_pred, alpha=0.6, s=20)
    lo, hi = float(min(y_te.min(), y_pred.min())), float(max(y_te.max(), y_pred.max()))
    ax.plot([lo, hi], [lo, hi], 'r--', alpha=0.5, label='perfect')
    ax.set_title(f'Test on {test_battery}  (RMSE={rmse:.2f}%, MAE={mae:.2f}%)')
    ax.set_xlabel('Actual SoH (%)'); ax.set_ylabel('Predicted SoH (%)')
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

metrics_df = pd.DataFrame(fold_metrics)
print('\n=== CV summary ===')
print(metrics_df.to_string(index=False))
print(f'\n  Mean RMSE: {metrics_df["rmse"].mean():.2f}%  ± {metrics_df["rmse"].std():.2f}')
print(f'  Mean MAE : {metrics_df["mae"].mean():.2f}%  ± {metrics_df["mae"].std():.2f}')

## Cell 8 — Final Train (All Data) + Export

Setelah CV menunjukkan model generalisasi OK, re-train pakai SEMUA 4 baterai dan export
`soh_xgb_model.json`. File ini langsung kompatibel dengan `jetson/src/main.py:55`:

```python
self.xgb_model = xgb.XGBRegressor()
self.xgb_model.load_model('models/weights/soh_xgb_model.json')
```

In [ ]:
final_model = xgb.XGBRegressor(**HYPERPARAMS)
final_model.fit(X, y)

OUT_PATH = 'soh_xgb_model.json'
final_model.save_model(OUT_PATH)
print(f'[OK] Saved {OUT_PATH}')
print(f'     Trained on {len(X)} cycles from {sorted(groups.unique())}')
print(f'     Features (order matters!): {FEATURES}')

# Sanity-check round-trip
reload = xgb.XGBRegressor()
reload.load_model(OUT_PATH)
sample = X.iloc[:3]
print(f'\n[sanity] reload.predict(first 3 rows) = {reload.predict(sample)}')
print(f'[sanity] original.predict(first 3 rows) = {final_model.predict(sample)}')

### Feature importance (final model)

In [ ]:
importance = pd.Series(final_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
ax = importance.plot.barh(figsize=(7, 3), color='C0')
ax.set_title('XGBoost feature importance (final model)')
ax.set_xlabel('importance'); ax.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()
print(importance.sort_values(ascending=False).to_string())

### Download model file

In [ ]:
try:
    from google.colab import files
    files.download(OUT_PATH)
except Exception as e:
    print(f'[!] files.download skipped (not in Colab?): {e}')

## Deploy

1. Download `soh_xgb_model.json` (otomatis di Colab pada sel sebelumnya)
2. Taruh di `jetson/models/weights/soh_xgb_model.json` di Jetson
3. Jalankan `python jetson/src/main.py` — log akan menampilkan `[AI] Loaded XGBoost SOH Model from ...`

Jika RMSE > 5% di CV, pertimbangkan:
- Tambah fitur (cycle count, discharge time) — butuh modifikasi STM32 firmware
- Hyperparameter tuning via `GridSearchCV` atau `Optuna`
- Cek apakah baterai 18650 di production mirip dengan NASA 18650 yang ditraining